# NeMo-Skills Competitive Programmer Finetuning

This notebook prepares the OpenCodeReasoning-2 dataset (CPP split) and finetunes a Qwen2.5-Coder-7B-Instruct model. Storage management is optimized to handle large base datasets.

In [ ]:
%pip install git+https://github.com/NVIDIA-NeMo/Skills.git
%pip install datasets==2.16.0

In [ ]:
!ns setup

## Step 1: Data Preparation

Choose whether to extract solutions from original datasets or generate them with an LLM.

In [ ]:
# Set this to True if you want to use existing solutions from the base datasets (e.g., apps, taco)
# Set to False if you want to generate synthetic solutions with a separate LLM step
USE_DATASET_SOLUTION = True

cmd = f"python /home/fax/Documents/yappathon/algorithms/gp/NeMo-Skills/recipes/opencodereasoning/scripts/prepare_questions.py \
    --output_dir /workspace/opencodereasoning/data"
if USE_DATASET_SOLUTION:
    cmd += " --use_dataset_solution"

get_ipython().system(cmd)

## Step 2: (Optional) Generate Solutions

If you set `USE_DATASET_SOLUTION = False`, you should run this stage.

In [ ]:
if not USE_DATASET_SOLUTION:
    from nemo_skills.pipeline.cli import generate, wrap_arguments
    generate(
        input_file="/workspace/opencodereasoning/data/open_code_reasoning_questions.jsonl",
        output_dir="/workspace/opencodereasoning/solutions",
        expname="ocr-solutions",
        cluster="slurm",
        model="Qwen/QwQ-32B-Preview",
        server_type="sglang",
        server_gpus=8,
        prompt_config="/home/fax/Documents/yappathon/algorithms/gp/NeMo-Skills/recipes/opencodereasoning/prompts/generate_cpp_soln.yaml",
        inference=dict(tokens_to_generate=8192, temperature=0.6)
    )
else:
    print("Skipping LLM solution generation. Using solutions from dataset.")

## Step 3: Prepare SFT Data

Apply formatting for finetuning.

In [ ]:
input_path = "/workspace/opencodereasoning/data/open_code_reasoning_questions.jsonl" if USE_DATASET_SOLUTION else "/workspace/opencodereasoning/solutions/output.jsonl"
!python -m nemo_skills.training.prepare_data \
    ++input_files={input_path} \
    ++output_path=/workspace/opencodereasoning/sft-data.jsonl \
    ++prompt_config=generic/math \
    ++add_unlabeled=true

## Step 4: Finetune Qwen2.5-Coder-7B-Instruct

Train using NeMo-RL.

In [ ]:
!ns nemo_rl sft \
    --cluster=slurm \
    --training_data=/workspace/opencodereasoning/sft-data.jsonl \
    --hf_model=Qwen/Qwen2.5-Coder-7B-Instruct \
    --backend=megatron \
    --expname=qwen-coder-sft \
    ++policy.max_total_sequence_length=8192 \
    ++sft.max_num_epochs=2